In [ ]:
!pip install -q torch torchvision transformers pycocotools scipy pillow matplotlib

In [ ]:
import contextlib
import io
import json
import random
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from scipy.optimize import linear_sum_assignment
from torchvision.ops import batched_nms, box_convert, box_iou, generalized_box_iou
from transformers import (
    Owlv2ForObjectDetection,
    Owlv2Processor,
    Sam3Model,
    Sam3Processor,
)

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
OWL_MODEL_NAME = "google/owlv2-base-patch16-ensemble"
# facebook/sam3 is gated (manual approval); this is an ungated mirror of the same files
SAM3_MODEL_NAME = "jetjodh/sam3"
DATA_DIR = Path("../data/lvis")
LVIS_URL = "https://dl.fbaipublicfiles.com/LVIS/lvis_v1_val.json.zip"
NUM_IMAGES = 20
SEED = 0

random.seed(SEED)
torch.manual_seed(SEED)
print(f"DEVICE = {DEVICE}")

# Open-Vocabulary Detection and Segmentation

A closed-set detector answers "where are the 80 things I was trained on?". Production vision now
asks "where is *this*?", where *this* is any noun phrase typed at run time. This notebook builds the
two mechanisms that make that work: **region–text scoring** (OWLv2) and **promptable concept
segmentation with a presence head** (SAM 3). It measures both on the rarest categories of LVIS.

---

# 1) Introduction

## 1.1 The problem

Given an image and a text prompt $k$ ("egg roll", "trampoline"), return every instance of $k$ as a
box, and ideally a mask, with a calibrated score. The prompt vocabulary is unbounded and unknown at
training time.

## 1.2 Where earlier approaches fail

| Approach | Mechanism | Failure |
|---|---|---|
| Closed-set (YOLO, DETR on COCO) | Fixed classifier over $C$ classes | Cannot output a class outside $C$, so AP on new classes is 0 by construction |
| Two-stage CLIP crop scoring | Propose boxes, embed each crop with CLIP | Slow (one CLIP pass per crop); CLIP was trained on whole images, so crop scores are poorly calibrated |
| Open-vocabulary detector (OWL, Grounding DINO) | Per-region embedding compared with a text embedding | Each region must decide alone whether the concept exists anywhere; hard negatives ("a *cougar*" in a photo of a house cat) produce confident false positives |
| Promptable segmenter, SAM 1/2 | Point/box prompt to mask | Needs a geometric prompt per object; no notion of a concept |

## 1.3 The core ideas

1. **Replace the classifier with a dot product.** A detector head emits an embedding $e_b$ per
   region. The class weights become text embeddings $t_k$, so any phrase is a class:
   $s_{b,k} = \sigma\big(\lambda_b \cos(e_b, t_k) + \beta_b\big)$.
2. **Predict a set, not a list.** DETR-style decoders are trained with one-to-one Hungarian
   matching, so each object gets one prediction and duplicates are penalised. NMS becomes optional.
3. **Factor "is it here?" out of "where is it?".** SAM 3 adds one global **presence token**:
   $p(\text{query } q \text{ is a } k) = p(k \text{ present}) \cdot p(q \text{ matches} \mid k \text{ present})$.
   Recognition is decided once per image, so localisation queries no longer need to.

## 1.4 Findings from the past year

| Model | Date | What is new |
|---|---|---|
| **SAM 3** (Meta) | Nov 2025, ICLR 2026 | *Promptable Concept Segmentation*: a noun phrase or image exemplar returns masks + instance IDs for **all** matches in images and video. Image-level detector and memory-based video tracker share one backbone; a **presence head** decouples "is the concept here" from "where". Data engine with ~4M unique concept labels incl. hard negatives; SA-Co benchmark; ~2× prior accuracy on PCS. Limits: no complex referring expressions; video cost grows with tracked objects |
| **SAM 2** (Meta) | 2024 | Streaming memory attention for video; the tracker SAM 3 builds on |
| **YOLO26** (Ultralytics) | late 2025 | Native end-to-end, **NMS-free** YOLO; drops Distribution Focal Loss; trained with the MuSGD optimizer (Muon-style). AGPL-3.0 |
| **RF-DETR** (Roboflow) | 2025 | Real-time DETR on a DINOv2 backbone; Apache-2.0; vendor reports it beats YOLO26 on COCO (treat as vendor claim) |
| **D-FINE / DEIM / DEIMv2** | Oct 2024 / CVPR 2025 / Sep 2025 | Fine-grained distribution refinement of box edges; DEIM improves the DETR matcher (dense one-to-one); DEIMv2 adds DINOv3 backbones. DEIM-D-FINE-X: 56.5 AP at 78 FPS on T4 |
| **Grounding DINO 1.5/1.6, YOLO-World, YOLOE, OWLv2, LLMDet** | 2024–25 | Text-prompted open-vocabulary detection; Grounding DINO is the accuracy reference, YOLO-World / YOLOE the real-time family |
| **Florence-2** (Microsoft) | 2024 | One seq2seq model for detection, grounding, captioning via task prompts |

## 1.5 What this notebook covers

| Section | Content |
|---|---|
| 2) Setup | LVIS v1 val, rare-category slice; OWLv2-base, SAM 3 |
| 3) The math | Region–text scoring, Hungarian set matching, presence factorisation, LVIS federated AP |
| 4) From scratch | OWLv2 scoring head (checked against the model), matching cost + assignment, presence scores, federated evaluation |
| 5) End-to-end | Both models on 20 images × 21 rare concepts: AP, masks, runtime, memory |
| 6) Variants | Presence head on/off (hard negatives); NMS vs no NMS on the same predictions |
| 7) Practical notes | Latency, licenses, prompts, fine-tuning |
| 8) Conclusion | Decision table |

---

# 2) Setup

## 2.1 Data: LVIS v1 rare categories

LVIS re-annotates COCO images with 1,203 categories split by training frequency: *frequent*,
*common* and **rare** (1–10 training images). Rare classes are the standard open-vocabulary test:
a detector cannot have memorised them. We sample 20 val images that contain at least one rare
object and prompt **every** rare concept present in any of them. Each image is therefore also
queried for concepts it does not contain.

LVIS is **federated**: for each image, only some categories are verified, either present
(annotated) or absent (`neg_category_ids`). A detection of category $k$ on an image where $k$ was
never checked is neither right nor wrong, so it is dropped before scoring.

In [ ]:
def load_lvis_val() -> dict:
    """Download (once) and parse the LVIS v1 val annotations."""
    path = DATA_DIR / "lvis_v1_val.json"
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        archive = DATA_DIR / "lvis_v1_val.json.zip"
        urllib.request.urlretrieve(LVIS_URL, archive)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(DATA_DIR)
    return json.loads(path.read_text())


start = time.time()
lvis = load_lvis_val()
categories = {c["id"]: c for c in lvis["categories"]}
anns_by_image = defaultdict(list)
for ann in lvis["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

rare_ids = {cid for cid, c in categories.items() if c["frequency"] == "r"}
rare_images = [
    im
    for im in lvis["images"]
    if any(a["category_id"] in rare_ids for a in anns_by_image[im["id"]])
]
images = random.sample(rare_images, NUM_IMAGES)
vocab = sorted(
    {
        a["category_id"]
        for im in images
        for a in anns_by_image[im["id"]]
        if a["category_id"] in rare_ids
    }
)
print(
    f"LVIS val: {len(lvis['images'])} images, {len(categories)} categories "
    f"({len(rare_ids)} rare) | loaded in {time.time() - start:.0f} s"
)
print(f"images with a rare object: {len(rare_images)}; sampled {len(images)}")
print(f"prompt vocabulary: {len(vocab)} rare concepts")

LVIS names carry WordNet-style disambiguation (`pin_(non_jewelry)`, `salmon_(food)`). Turned
into plain phrases they become usable text prompts.

In [ ]:
def to_phrase(name: str) -> str:
    """'pin_(non_jewelry)' -> 'pin non jewelry'."""
    return name.replace("_", " ").replace("(", "").replace(")", "")


prompts = [to_phrase(categories[cid]["name"]) for cid in vocab]
print(prompts)

pil_images = {}
(DATA_DIR / "images").mkdir(exist_ok=True)
for im in images:
    path = DATA_DIR / "images" / Path(im["coco_url"]).name
    if not path.exists():
        urllib.request.urlretrieve(im["coco_url"], path)
    pil_images[im["id"]] = Image.open(path).convert("RGB")

# Ground truth restricted to the prompted concepts
gt_anns = [
    a for im in images for a in anns_by_image[im["id"]] if a["category_id"] in vocab
]
# Per image: concepts verified present or absent (the federated "checked" set)
checked = {
    im["id"]: {a["category_id"] for a in anns_by_image[im["id"]]}
    | set(im["neg_category_ids"])
    for im in images
}
print(f"{len(gt_anns)} ground-truth instances of the prompted concepts")
num_checked = sum(len(checked[im["id"]] & set(vocab)) for im in images)
print(
    f"(image, concept) pairs checked: {num_checked} "
    f"of {len(images) * len(vocab)} prompted"
)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, im in zip(axes.flat, images[:8]):
    ax.imshow(pil_images[im["id"]])
    for ann in anns_by_image[im["id"]]:
        if ann["category_id"] in vocab:
            x, y, w, h = ann["bbox"]
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="lime", lw=2))
            ax.text(
                x,
                y,
                to_phrase(categories[ann["category_id"]]["name"]),
                color="black",
                backgroundcolor="lime",
                fontsize=7,
            )
    ax.axis("off")
plt.suptitle("Rare-category ground truth")
plt.tight_layout()
plt.show()

## 2.2 Models

| Model | Params | Architecture | Prompt |
|---|---|---|---|
| OWLv2 base/16 | ~155 M | CLIP ViT-B/16; every output patch token is a box candidate with an embedding | text, or an image exemplar |
| SAM 3 | ~840 M | Perception-Encoder ViT + DETR encoder/decoder with 200 queries + presence token + mask decoder | noun phrase, exemplar boxes, or points |

SAM 3 runs in fp16 to keep its memory near 3 GB.

In [ ]:
def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


start = time.time()
owl = Owlv2ForObjectDetection.from_pretrained(OWL_MODEL_NAME).to(DEVICE).eval()
owl_processor = Owlv2Processor.from_pretrained(OWL_MODEL_NAME)
sam3 = Sam3Model.from_pretrained(SAM3_MODEL_NAME, dtype=torch.float16).to(DEVICE).eval()
sam3_processor = Sam3Processor.from_pretrained(SAM3_MODEL_NAME)
print(
    f"OWLv2 {sum(p.numel() for p in owl.parameters()) / 1e6:.0f} M params | "
    f"SAM 3 {sum(p.numel() for p in sam3.parameters()) / 1e6:.0f} M params"
)
print(f"load {time.time() - start:.0f} s | device memory {device_memory_gb():.2f} GB")

---

# 3) The math

## 3.1 Region–text scoring (OWL)

The image encoder outputs one feature $h_b \in \mathbb{R}^{d_v}$ per patch $b$. Every patch is a
box candidate. A text encoder maps prompt $k$ to $t_k \in \mathbb{R}^{d_t}$. With a learned
projection $W$:

$$
e_b = W h_b,\qquad
\ell_{b,k} = \underbrace{\big(\operatorname{elu}(w_s^\top h_b) + 1\big)}_{\lambda_b > 0}\Big(\frac{e_b^\top t_k}{\lVert e_b\rVert\,\lVert t_k\rVert} + \underbrace{w_m^\top h_b}_{\beta_b}\Big),\qquad
s_{b,k} = \sigma(\ell_{b,k}).
$$

Cosine similarity alone is a poor detection score: background patches can resemble the text too.
The **per-box** scale $\lambda_b$ and shift $\beta_b$, computed from the patch itself, let the
model learn that some patches are object-like and others are not, independently of the query.
Training uses a sigmoid focal loss per (box, query) pair, so scores are independent across prompts
and do not compete in a softmax.

## 3.2 Set prediction and Hungarian matching (DETR, SAM 3)

A decoder emits $Q$ predictions $\{(\hat p_q, \hat b_q)\}$ for $N \le Q$ ground-truth boxes
$\{b_i\}$. Training first finds the one-to-one assignment $\sigma$ of minimum cost

$$
\sigma^\star = \arg\min_{\sigma} \sum_{i=1}^{N} \Big( -\lambda_{\text{cls}}\, \hat p_{\sigma(i)}
+ \lambda_{L1} \lVert b_i - \hat b_{\sigma(i)} \rVert_1
- \lambda_{\text{giou}}\, \operatorname{GIoU}(b_i, \hat b_{\sigma(i)}) \Big),
$$

solved exactly by the Hungarian algorithm in $O(Q^3)$. Matched queries are trained toward their
box and label. **All unmatched queries are trained toward "no object"**, including a second query
sitting on the same object. Duplicates are therefore penalised during training, which is why
these models can skip NMS. GIoU extends IoU with a term that stays informative when boxes do not
overlap:

$$
\operatorname{GIoU}(A, B) = \operatorname{IoU}(A, B) - \frac{|C \setminus (A \cup B)|}{|C|},
\quad C = \text{smallest box enclosing } A \text{ and } B.
$$

## 3.3 The presence factorisation (SAM 3)

Without a presence token, every query must solve two problems at once: *is this concept anywhere
in the image?* (global recognition) and *is it this region?* (local). SAM 3 adds one learned token
$z$ to the decoder that attends to the whole image and text, and factorises

$$
s_q = \underbrace{\sigma(\ell^{\text{pres}})}_{p(k \text{ in image})} \cdot \underbrace{\sigma(\ell_q)}_{p(q \mid k \text{ present})},
\qquad \ell_q = \frac{(W_q x_q)^\top (W_t \bar t)}{\sqrt{d}},
$$

where $x_q$ is query $q$'s decoder state and $\bar t$ the mean-pooled prompt tokens. On a **hard
negative**, where the concept is absent but something similar is present, the per-query term can
still be high. The presence term is the single place where the model decides "absent", and it
suppresses every query at once.

## 3.4 Evaluation: federated AP

AP for concept $k$ is the area under the precision–recall curve of its detections, sorted by score;
a detection is a true positive if IoU $\ge \tau$ with an unmatched ground truth. COCO-style AP
averages over $\tau \in \{0.50, 0.55, \dots, 0.95\}$, and AP50 uses $\tau = 0.5$. LVIS evaluates
$k$ only on images where $k$ was verified (§2.1), and averages AP over concepts.

---

# 4) From-scratch implementation

## 4.1 The OWLv2 region–text head

Rebuilt from the model's own weights (`class_head.dense0` is $W$, `logit_scale` is $w_s$,
`logit_shift` is $w_m$) and checked against the logits the model returns.

In [ ]:
def region_text_logits(
    patch_feats: torch.Tensor, text_embeds: torch.Tensor, head: torch.nn.Module
) -> torch.Tensor:
    """patch_feats (num_boxes, d_v), text_embeds (num_prompts, d_t).

    Returns logits (num_boxes, num_prompts).
    """
    box_embeds = head.dense0(patch_feats)
    box_embeds = box_embeds / (box_embeds.norm(dim=-1, keepdim=True) + 1e-6)
    text_embeds = text_embeds / (text_embeds.norm(dim=-1, keepdim=True) + 1e-6)
    cosine = box_embeds @ text_embeds.T
    # per-box shift beta_b and positive scale lambda_b, both read off the patch feature
    shift = head.logit_shift(patch_feats)
    scale = torch.nn.functional.elu(head.logit_scale(patch_feats)) + 1
    return (cosine + shift) * scale


@torch.no_grad()
def owl_forward(image: Image.Image) -> dict:
    """One OWLv2 pass over all prompts; returns the raw outputs on DEVICE."""
    inputs = owl_processor(text=[prompts], images=image, return_tensors="pt").to(DEVICE)
    return owl(**inputs)


sample = images[0]
owl_out = owl_forward(pil_images[sample["id"]])
# (1, grid, grid, d_v) -> (num_boxes, d_v)
patch_feats = owl_out.image_embeds.flatten(1, 2)[0]
ours = region_text_logits(patch_feats, owl_out.text_embeds[0], owl.class_head)
print(
    f"patch features {tuple(patch_feats.shape)}  "
    f"text embeddings {tuple(owl_out.text_embeds[0].shape)}"
)
print(
    f"logits {tuple(ours.shape)} (num_boxes, num_prompts); "
    f"max |ours - model| = {(ours - owl_out.logits[0]).abs().max():.2e}"
)

How much do the per-box terms matter? Rank the boxes for each concept present in this image by
plain cosine and by the full logit, and compare the best match with the ground truth.

In [ ]:
def owl_boxes_xyxy(pred_boxes: torch.Tensor, image: Image.Image) -> torch.Tensor:
    """OWLv2 normalised (cx, cy, w, h) on the padded square -> pixel xyxy."""
    return box_convert(pred_boxes, "cxcywh", "xyxy") * max(image.size)


def gt_boxes_xyxy(image_id: int, category_id: int) -> torch.Tensor:
    """LVIS ground-truth boxes of one concept in one image, (num_gt, 4) xyxy pixels."""
    boxes = [
        a["bbox"] for a in anns_by_image[image_id] if a["category_id"] == category_id
    ]
    return box_convert(torch.tensor(boxes, dtype=torch.float32), "xywh", "xyxy")


head = owl.class_head
box_embeds = head.dense0(patch_feats)
text = owl_out.text_embeds[0]
cosine = (box_embeds / box_embeds.norm(dim=-1, keepdim=True)) @ (
    text / text.norm(dim=-1, keepdim=True)
).T
boxes = owl_boxes_xyxy(owl_out.pred_boxes[0].cpu(), pil_images[sample["id"]])

for cid in sorted({a["category_id"] for a in anns_by_image[sample["id"]]} & set(vocab)):
    k = vocab.index(cid)
    gt = gt_boxes_xyxy(sample["id"], cid)
    for name, scores in [("cosine only", cosine[:, k]), ("full logit ", ours[:, k])]:
        top = scores.argmax().item()
        best_iou = box_iou(boxes[top : top + 1], gt).max().item()
        print(
            f"{prompts[k]:15s} {name}: top box IoU with ground truth = {best_iou:.2f}"
        )

## 4.2 Matching cost and one-to-one assignment

The cost of §3.2 is built from scratch. GIoU comes from `torchvision`, and the Hungarian solver from
SciPy. Applied to SAM 3's 200 queries for a concept with several instances, it shows which queries
training would treat as the objects. Everything else counts as "no object".

In [ ]:
def matching_cost(
    pred_scores: torch.Tensor,
    pred_boxes: torch.Tensor,
    gt_boxes: torch.Tensor,
    image_size: tuple[int, int],
    w_cls: float = 2.0,
    w_l1: float = 5.0,
    w_giou: float = 2.0,
) -> torch.Tensor:
    """pred_scores (Q,), pred_boxes (Q, 4), gt_boxes (N, 4) xyxy pixels.

    Returns the matching cost (N, Q).
    """
    width, height = image_size
    norm = torch.tensor([width, height, width, height])
    # L1 on normalised coordinates, so its scale does not depend on image size
    l1 = torch.cdist(gt_boxes / norm, pred_boxes / norm, p=1)
    giou = generalized_box_iou(gt_boxes, pred_boxes)
    return -w_cls * pred_scores[None, :] + w_l1 * l1 - w_giou * giou


def hungarian_match(cost: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    """cost (N, Q) -> (gt indices, matched query indices), each of length N."""
    return linear_sum_assignment(cost.numpy())


@torch.no_grad()
def sam3_concept(vision_embeds: dict, prompt_ids: dict, k: int) -> dict:
    """Run SAM 3's detector for prompt k on cached vision features.

    Outputs are moved to CPU as float32.
    """
    out = sam3(
        vision_embeds=vision_embeds,
        input_ids=prompt_ids["input_ids"][k : k + 1],
        attention_mask=prompt_ids["attention_mask"][k : k + 1],
    )
    return {
        "boxes": out.pred_boxes[0].float().cpu(),
        "query_logits": out.pred_logits[0].float().cpu(),
        "presence_logit": out.presence_logits[0].float().cpu(),
        "masks": out.pred_masks[0].float().cpu(),
    }

Test on the (image, concept) pair with the most instances: match its ground-truth boxes to SAM 3's
queries and see where the matched queries rank by score.

In [ ]:
prompt_ids = sam3_processor(text=prompts, return_tensors="pt").to(DEVICE)
multi = max(
    ((im, cid) for im in images for cid in vocab),
    key=lambda pair: sum(
        a["category_id"] == pair[1] for a in anns_by_image[pair[0]["id"]]
    ),
)
image, cid = pil_images[multi[0]["id"]], multi[1]
k = vocab.index(cid)
pixel_values = (
    sam3_processor(images=image, return_tensors="pt").to(DEVICE).pixel_values.half()
)
with torch.no_grad():
    vision_embeds = sam3.get_vision_features(pixel_values=pixel_values)
res = sam3_concept(vision_embeds, prompt_ids, k)
width, height = image.size
pred_boxes = res["boxes"] * torch.tensor([width, height, width, height])
query_scores = res["query_logits"].sigmoid()
gt = gt_boxes_xyxy(multi[0]["id"], cid)

gt_idx, query_idx = hungarian_match(
    matching_cost(query_scores, pred_boxes, gt, image.size)
)
print(f"concept '{prompts[k]}': {len(gt)} instances, {len(query_scores)} queries")
for i, q in zip(gt_idx, query_idx):
    print(
        f"  gt {i} -> query {q:3d}  score {query_scores[q]:.2f}  "
        f"IoU {box_iou(gt[i : i + 1], pred_boxes[q : q + 1]).item():.2f}"
    )
rank = query_scores.argsort(descending=True).argsort()
print(f"rank of matched queries by score: {sorted(rank[query_idx].tolist())}")
print(f"queries scoring > 0.5: {(query_scores > 0.5).sum().item()}")

## 4.3 Presence-weighted scores

The final SAM 3 score per query is $\sigma(\ell^{\text{pres}})\,\sigma(\ell_q)$ (§3.3). Comparing a
concept that is present with one that is absent from the same image shows which factor does the
work.

In [ ]:
def concept_scores(res: dict, use_presence: bool = True) -> torch.Tensor:
    """(num_queries,) final query scores, with or without the presence factor."""
    scores = res["query_logits"].sigmoid()
    return scores * res["presence_logit"].sigmoid() if use_presence else scores


absent_k = next(
    j
    for j, c in enumerate(vocab)
    if c not in checked[multi[0]["id"]] or c in multi[0]["neg_category_ids"]
)
absent = sam3_concept(vision_embeds, prompt_ids, absent_k)
for name, r in [
    (prompts[k] + " (present)", res),
    (prompts[absent_k] + " (absent)", absent),
]:
    print(
        f"{name:28s} presence {r['presence_logit'].sigmoid().item():.3f} | "
        f"max query {concept_scores(r, False).max():.3f} | "
        f"max final {concept_scores(r).max():.3f}"
    )

## 4.4 Federated AP

A thin wrapper around `pycocotools`. Detections are first filtered to (image, concept) pairs that
LVIS verified, then up to 100 per image are kept and the COCO protocol runs on the rest.

In [ ]:
def federated_ap(detections: list[dict]) -> tuple[float, float]:
    """detections: dicts with image_id, category_id, bbox (xywh), score.

    Returns (AP, AP50).
    """
    detections = [d for d in detections if d["category_id"] in checked[d["image_id"]]]
    per_image = defaultdict(list)
    for d in detections:
        per_image[d["image_id"]].append(d)
    detections = [
        d
        for dets in per_image.values()
        for d in sorted(dets, key=lambda d: -d["score"])[:100]
    ]
    gt = {
        "images": images,
        "annotations": [dict(a, iscrowd=0) for a in gt_anns],
        "categories": [categories[c] for c in vocab],
    }
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO()
        coco_gt.dataset = gt
        coco_gt.createIndex()
        coco_eval = COCOeval(coco_gt, coco_gt.loadRes(detections), "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
    return float(coco_eval.stats[0]), float(coco_eval.stats[1])


def to_detections(
    image_id: int, boxes: torch.Tensor, scores: torch.Tensor, labels: torch.Tensor
) -> list[dict]:
    """xyxy pixel boxes + scores + vocab indices -> COCO-format detections."""
    xywh = box_convert(boxes, "xyxy", "xywh")
    return [
        {"image_id": image_id, "category_id": vocab[label], "bbox": box, "score": score}
        for box, score, label in zip(xywh.tolist(), scores.tolist(), labels.tolist())
    ]


perfect = [
    {
        "image_id": a["image_id"],
        "category_id": a["category_id"],
        "bbox": a["bbox"],
        "score": 1.0,
    }
    for a in gt_anns
]
print(f"ground truth as detections: AP {federated_ap(perfect)[0]:.3f}")

---

# 5) End-to-end on 20 images × 21 rare concepts

## 5.1 OWLv2

One forward pass per image scores all 21 prompts against all 3,600 patch boxes.

In [ ]:
start = time.time()
owl_raw = {}
for im in images:
    out = owl_forward(pil_images[im["id"]])
    owl_raw[im["id"]] = (
        owl_boxes_xyxy(out.pred_boxes[0].cpu(), pil_images[im["id"]]),
        out.logits[0].sigmoid().cpu(),
    )
owl_seconds = time.time() - start


def owl_detections(
    nms_iou: float | None = None, min_score: float = 0.001
) -> list[dict]:
    """Every (box, concept) pair above min_score, optionally class-wise NMS."""
    detections = []
    for image_id, (boxes, probs) in owl_raw.items():
        num_boxes, num_prompts = probs.shape
        scores = probs.flatten()
        labels = torch.arange(num_prompts).repeat(num_boxes)
        all_boxes = boxes.repeat_interleave(num_prompts, dim=0)
        keep = scores > min_score
        all_boxes, scores, labels = all_boxes[keep], scores[keep], labels[keep]
        if nms_iou is not None:
            keep = batched_nms(all_boxes, scores, labels, nms_iou)
            all_boxes, scores, labels = all_boxes[keep], scores[keep], labels[keep]
        detections += to_detections(image_id, all_boxes, scores, labels)
    return detections


owl_ap, owl_ap50 = federated_ap(owl_detections())
print(
    f"OWLv2: AP {owl_ap:.3f} | AP50 {owl_ap50:.3f} | "
    f"{owl_seconds / NUM_IMAGES:.2f} s/image | "
    f"device memory {device_memory_gb():.2f} GB"
)

## 5.2 SAM 3

The vision backbone runs once per image. The DETR encoder/decoder runs once per prompt on the
cached features, so the cost grows with the number of concepts.

In [ ]:
start = time.time()
sam3_raw = {}
for im in images:
    image = pil_images[im["id"]]
    pixel_values = (
        sam3_processor(images=image, return_tensors="pt").to(DEVICE).pixel_values.half()
    )
    with torch.no_grad():
        vision_embeds = sam3.get_vision_features(pixel_values=pixel_values)
    width, height = image.size
    results = []
    for k in range(len(prompts)):
        res = sam3_concept(vision_embeds, prompt_ids, k)
        res["boxes"] = res["boxes"] * torch.tensor([width, height, width, height])
        # masks are only needed for the figure below
        res["masks"] = res["masks"] if im["id"] == images[0]["id"] else None
        results.append(res)
    sam3_raw[im["id"]] = results
sam3_seconds = time.time() - start


def sam3_detections(
    use_presence: bool = True, nms_iou: float | None = None
) -> list[dict]:
    """All queries of all prompts as detections, scored with or without presence."""
    detections = []
    for image_id, results in sam3_raw.items():
        boxes = torch.cat([r["boxes"] for r in results])
        scores = torch.cat([concept_scores(r, use_presence) for r in results])
        labels = torch.arange(len(results)).repeat_interleave(len(results[0]["boxes"]))
        if nms_iou is not None:
            keep = batched_nms(boxes, scores, labels, nms_iou)
            boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
        detections += to_detections(image_id, boxes, scores, labels)
    return detections


sam3_ap, sam3_ap50 = federated_ap(sam3_detections())
print(
    f"SAM 3: AP {sam3_ap:.3f} | AP50 {sam3_ap50:.3f} | "
    f"{sam3_seconds / NUM_IMAGES:.2f} s/image ({len(prompts)} prompts) | "
    f"device memory {device_memory_gb():.2f} GB"
)

## 5.3 Masks

SAM 3 returns a mask for every query. Here are the concepts present in the first image, keeping
queries whose final score exceeds 0.3 (or the single best query if none does).

In [ ]:
im = images[0]
image = pil_images[im["id"]]
present = sorted({a["category_id"] for a in anns_by_image[im["id"]]} & set(vocab))
fig, axes = plt.subplots(1, len(present) + 1, figsize=(5 * (len(present) + 1), 5))
axes = np.atleast_1d(axes)
axes[0].imshow(image)
axes[0].set_title("image")
for ax, cid in zip(axes[1:], present):
    res = sam3_raw[im["id"]][vocab.index(cid)]
    scores = concept_scores(res)
    keep = scores >= min(0.3, scores.max().item())
    masks = (
        torch.nn.functional.interpolate(
            res["masks"][keep][None], size=image.size[::-1], mode="bilinear"
        )[0]
        > 0
    )
    overlay = np.array(image) / 255.0
    for mask in masks:
        overlay[mask.numpy()] = 0.5 * overlay[mask.numpy()] + 0.5 * np.random.rand(3)
    ax.imshow(overlay)
    ax.set_title(
        f"'{to_phrase(categories[cid]['name'])}': {keep.sum().item()} instances"
    )
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---

# 6) Variants

## 6.1 Presence head on and off

Two views of the same predictions. The first is AP under the federated protocol. The second is a
direct hard-negative test: for every (image, concept) pair where the concept is **absent**, does
any query still score above 0.5?

In [ ]:
print("scoring            |   AP  | AP50 | absent pairs with a query > 0.5")
for use_presence in [True, False]:
    ap, ap50 = federated_ap(sam3_detections(use_presence))
    false_alarms, absent_pairs = 0, 0
    for im in images:
        present = {a["category_id"] for a in anns_by_image[im["id"]]}
        for k, cid in enumerate(vocab):
            if cid not in present:
                absent_pairs += 1
                false_alarms += int(
                    concept_scores(sam3_raw[im["id"]][k], use_presence).max() > 0.5
                )
    label = "query x presence" if use_presence else "query only      "
    print(
        f"{label}   | {ap:.3f} | {ap50:.3f} | "
        f"{false_alarms}/{absent_pairs} ({false_alarms / absent_pairs:.1%})"
    )

## 6.2 NMS on set predictors

Both models were trained with one-to-one assignment (OWLv2 also uses DETR-style bipartite
matching). If that taught them not to emit duplicates, class-wise NMS should change little.

In [ ]:
print("model  | NMS IoU | AP    | AP50")
for nms_iou in [None, 0.5, 0.3]:
    ap, ap50 = federated_ap(owl_detections(nms_iou))
    print(f"OWLv2  | {str(nms_iou):7s} | {ap:.3f} | {ap50:.3f}")
for nms_iou in [None, 0.5, 0.3]:
    ap, ap50 = federated_ap(sam3_detections(True, nms_iou))
    print(f"SAM 3  | {str(nms_iou):7s} | {ap:.3f} | {ap50:.3f}")

---

# 7) Practical notes

**Cost scales differently.** OWLv2 scores every prompt in one pass (text embeddings are a matrix
multiply), so 1,000 prompts cost about the same as 10. SAM 3 reruns its DETR encoder/decoder per
prompt, so cost grows linearly with the vocabulary. Use it for a handful of concepts per image, or
put a cheap open-vocabulary filter in front.

**Prompts.** Short noun phrases work best. SAM 3 explicitly does not handle long referring
expressions ("the second cup from the left"); route those through a VLM that outputs a box, then
prompt SAM with the box. Disambiguate polysemous words ("pin" → "sewing pin") as LVIS names do.
When text fails, an **image exemplar** (a box around one instance) is the most reliable prompt for
both models.

**Thresholds.** Scores are per-prompt sigmoids, not a softmax, so one threshold does not transfer
across concepts. Calibrate per concept on a few labelled images if the vocabulary is fixed.

**Licenses.** OWLv2 is Apache-2.0. SAM 3 ships under Meta's SAM License, so check commercial terms.
YOLO26 is AGPL-3.0, which is copyleft over the network; RF-DETR and D-FINE/DEIM are Apache-2.0.

**When to fine-tune.** If the concept list is fixed and latency matters, distil into a real-time
closed-set detector (RF-DETR, DEIM, YOLO26), using SAM 3 as the labeller. Data engines in 2025–26
do exactly this. Keep open-vocabulary models for the long tail and for bootstrapping labels.

**Video.** SAM 3 pairs this detector with SAM 2's memory tracker to keep instance IDs across
frames; the cost grows with the number of tracked objects.

---

# 8) Conclusion

| Need | Pick | Why |
|---|---|---|
| Fixed classes, real-time | RF-DETR / DEIM / YOLO26 (closed-set) | Fastest, NMS-free, but cannot name new classes |
| Many arbitrary text queries per image | OWLv2 / Grounding DINO / YOLOE | All prompts in one pass; region–text dot product |
| Few concepts, need every instance + masks | SAM 3 | Presence head suppresses hard negatives; masks and video IDs |
| Long referring expressions | VLM to box, then SAM | Concept models handle short noun phrases only |
| Label a new dataset | SAM 3 as annotator, then train a closed-set model | Long-tail recall at labelling time, speed at deployment |